# 01 — Limpieza del dataset MDI 2019–2025

Este notebook realiza las siguientes operaciones:

- Lee el CSV del Ministerio del Interior (MDI) 2019–2025
- Limpia nombres de columnas y valores
- Corrige códigos ICCS mal formados presentes en el dataset original
- Construye el catálogo `codigo_iccs → delitos` desde `presunta_infraccion`
- **Imputa los registros `SIN_DATO` de `codigo_iccs` usando Random Forest** entrenado sobre variables de evento
- Crea la columna `delitos` con la descripción del tipo de delito
- Crea la columna `tipo_dato` (ORIGINAL / SINTETICO)
- **Imputa coordenadas `NO_APLICA` usando el centroide del subcircuito**
- Crea la columna `flag_coord` (REAL / IMPUTADO) para trazabilidad geoespacial
- Exporta `dataset_limpio.csv` a la carpeta de procesados


In [43]:
# Descomenta si trabajas en Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

In [44]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [45]:
# =========================
# RUTAS
# Ajusta archivo_entrada y archivo_salida según tu entorno
# =========================
archivo_entrada = r"C:\Users\jhono\Downloads\Proyecto Titulación\CSV\mdi_detenidosaprehendidos_guayaquil_2019_2025 (1) - copia.csv"
archivo_salida  = r"C:\Users\jhono\Downloads\Proyecto Titulación\CSV\mdi_detenidosaprehendidos_guayaquil_2019_2025_con_delitos_y_tipo_dato.csv"

guardar_catalogo  = True
archivo_catalogo  = r"C:\Users\jhono\Downloads\Proyecto Titulación\CSV\catalogo_iccs_generado.csv"

## 1. Lectura y limpieza inicial

In [46]:
# =========================
# LEER EL CSV
# =========================
df = pd.read_csv(
    archivo_entrada,
    sep=";",
    dtype=str,
    encoding="utf-8",
    keep_default_na=False
)

print("Filas:", len(df))
print("Columnas originales:")
print(df.columns.tolist())

Filas: 76860
Columnas originales:
['Unnamed: 0', ' codigo_iccs ', '  tipo', 'estado civil', ' estatus_migratorio', ' edad', 'sexo', '  genero', 'nacionalidad', 'autoidentificacion_etnica', 'nivel_de_instruccion', 'condicion', 'movilizacion', 'tipo_arma', 'arma', 'fecha_detencion_aprehension', 'hora_detencion_aprehension', 'lugar', 'tipo_lugar', 'nombre_zona', 'nombre_subzona', 'codigo_distrito', 'nombre_distrito', 'codigo_circuito', 'nombre_circuito', 'codigo_subcircuito', 'nombre_subcircuito', 'codigo_provincia', 'nombre_provincia', 'codigo_canton', 'nombre_canton', 'codigo_parroquia', 'nombre_parroquia', 'presunta_infraccion', 'latitud', 'longitud']


In [47]:
# =========================
# LIMPIAR NOMBRES DE COLUMNAS
# =========================
df.columns = [c.strip() for c in df.columns]

columnas_limpias = {c.strip().replace(" ", "_").lower(): c for c in df.columns}

col_codigo  = columnas_limpias["codigo_iccs"]
col_presunta = columnas_limpias["presunta_infraccion"]

print("Columna código encontrada:", col_codigo)
print("Columna presunta_infraccion encontrada:", col_presunta)

Columna código encontrada: codigo_iccs
Columna presunta_infraccion encontrada: presunta_infraccion


In [48]:
# =========================
# LIMPIAR VALORES
# =========================
df[col_codigo]   = df[col_codigo].astype(str).str.strip()
df[col_presunta] = df[col_presunta].astype(str).str.strip()

# Guardar valor original para construir tipo_dato al final
df["codigo_iccs_original"] = df[col_codigo]

## 2. Corrección de códigos ICCS mal formados

In [49]:
# =========================
# CORREGIR CÓDIGOS ICCS MAL FORMADOS EN EL DATASET ORIGINAL
# Estos códigos vienen con dígitos faltantes desde la fuente (MDI).
# La corrección se infiere a partir de presunta_infraccion.
# =========================
correccion_codigos = {
    "4011":     "040121.03",  # DELITOS CONTRA EL DERECHO A LA PROPIEDAD (5938 filas)
    "20111":    "020111.11",  # DELITOS CONTRA LA INTEGRIDAD PERSONAL (556 filas)
    "50211":    "050260.01",  # DELITOS CONTRA EL DERECHO A LA PROPIEDAD (519 filas)
    "70232":    "070231.03",  # DELITOS CONTRA LA FE PÚBLICA (428 filas)
    "02089.02": "020190.01",  # DELITOS DE VIOLENCIA CONTRA LA MUJER O NÚCLEO FAMILIAR (235 filas)
    "70320":    "070330.02",  # DELITOS CONTRA LA EFICIENCIA DE LA ADM. PÚBLICA (80 filas)
    "70311":    "070330.03",  # DELITOS CONTRA LA EFICIENCIA DE LA ADM. PÚBLICA (35 filas)
    "30210":    "030221.01",  # DIVERSAS FORMAS DE EXPLOTACIÓN (10 filas)
    "20321":    "020121.00",  # DIVERSAS FORMAS DE EXPLOTACIÓN (3 filas)
    "80322":    "080610.07",  # DELITO DE ODIO (2 filas)
}

filas_antes = df[col_codigo].isin(correccion_codigos).sum()
df[col_codigo] = df[col_codigo].replace(correccion_codigos)
filas_despues = df[col_codigo].isin(correccion_codigos).sum()
print(f"Códigos corregidos: {filas_antes} filas -> {filas_despues} sin corregir")

# El valor TRUE es un error de exportación del sistema MDI.
# Se trata como SIN_DATO para ser imputado por el modelo.
filas_true = (df[col_codigo].str.upper() == "TRUE").sum()
df.loc[df[col_codigo].str.upper() == "TRUE", col_codigo] = "SIN_DATO"
print(f"Filas TRUE convertidas a SIN_DATO: {filas_true}")

# Verificación: mostrar si quedan códigos fuera del formato estándar NNNNNN.NN
patron_iccs = re.compile(r'^\d{6}\.\d{2}$')
invalidos_restantes = df[
    ~df[col_codigo].str.upper().isin({"SIN_DATO", "SIN DATOS", "", "NAN", "NONE"}) &
    ~df[col_codigo].apply(lambda x: bool(patron_iccs.match(str(x))))
][col_codigo].value_counts()

if len(invalidos_restantes):
    print("\nCódigos aún fuera de formato:")
    print(invalidos_restantes)
else:
    print("\nVerificación OK: no quedan códigos con formato incorrecto.")

Códigos corregidos: 7806 filas -> 0 sin corregir
Filas TRUE convertidas a SIN_DATO: 1368

Verificación OK: no quedan códigos con formato incorrecto.


## 3. Construcción del catálogo ICCS → delitos

In [50]:
# =========================
# DEFINIR VALORES INVÁLIDOS Y MÁSCARA SIN_DATO
# =========================
invalidos_catalogo = {"SIN_DATO", "SIN DATOS", "TRUE", "", "NAN", "NONE"}
mask_sin_dato = df[col_codigo].str.upper().isin({"SIN_DATO", "SIN DATOS"})

print("Cantidad de SIN_DATO / SIN DATOS:", mask_sin_dato.sum())

Cantidad de SIN_DATO / SIN DATOS: 22183


In [51]:
# =========================
# CREAR CATÁLOGO codigo_iccs -> delitos
# usando la presunta_infraccion más frecuente por código
# =========================
df_validos = df[
    ~df[col_codigo].str.upper().isin(invalidos_catalogo) &
    (df[col_presunta] != "")
].copy()

catalogo = (
    df_validos
    .groupby(col_codigo)[col_presunta]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
catalogo.columns = ["codigo_iccs", "delitos"]

print("Cantidad de códigos válidos encontrados:", len(catalogo))
catalogo.head()

Cantidad de códigos válidos encontrados: 180


,codigo_iccs,delitos
0,010101.01,DELITOS CONTRA LA INVIOLABILIDAD DE LA VIDA
1,010102.01,DELITOS CONTRA LA INVIOLABILIDAD DE LA VIDA
2,010102.02,DELITOS CONTRA LA INVIOLABILIDAD DE LA VIDA
3,010102.03,DELITOS CONTRA EL DERECHO A LA PROPIEDAD
4,010102.04,DELITOS CONTRA LA INTEGRIDAD SEXUAL Y REPRODUC...


In [52]:
# =========================
# CREAR DICCIONARIO DE MAPEO
# =========================
mapa_delitos = dict(zip(catalogo["codigo_iccs"], catalogo["delitos"]))

## 4. Imputación de `codigo_iccs` con Random Forest

Los ~20.815 registros con `SIN_DATO` en `codigo_iccs` se imputan utilizando un clasificador
Random Forest entrenado sobre los ~56.000 registros con código válido.  
Las variables predictoras son exclusivamente de carácter contextual del evento
(zona geográfica, horario, tipo de arma, tipo de lugar), sin incluir ninguna variable
demográfica del detenido, en coherencia con el objetivo geoespacial del proyecto.

In [53]:
# =========================
# PREPARACIÓN DE VARIABLES PARA EL MODELO
# =========================

# --- Extraer variables temporales ---
df["_fecha_dt"] = pd.to_datetime(
    df["fecha_detencion_aprehension"], dayfirst=True, errors="coerce"
)
df["_hora_num"] = pd.to_numeric(
    df["hora_detencion_aprehension"].str.split(":").str[0], errors="coerce"
).fillna(-1).astype(int)
df["_dia_semana"] = df["_fecha_dt"].dt.dayofweek.fillna(-1).astype(int)
df["_mes"]        = df["_fecha_dt"].dt.month.fillna(-1).astype(int)
df["_anio"]       = df["_fecha_dt"].dt.year.fillna(-1).astype(int)

# --- Columnas predictoras (solo evento, sin demografía) ---
FEATURES = [
    "tipo_arma",
    "tipo_lugar",
    "nombre_zona",
    "codigo_distrito",
    "codigo_circuito",
    "codigo_subcircuito",
    "_hora_num",
    "_dia_semana",
    "_mes",
    "_anio",
]

# --- Codificar variables categóricas ---
encoders = {}
df_model = df[FEATURES].copy()

for col in ["tipo_arma", "tipo_lugar", "nombre_zona",
            "codigo_distrito", "codigo_circuito", "codigo_subcircuito"]:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le

print("Variables listas. Shape:", df_model.shape)
print("Features usadas:", FEATURES)

Variables listas. Shape: (76860, 10)
Features usadas: ['tipo_arma', 'tipo_lugar', 'nombre_zona', 'codigo_distrito', 'codigo_circuito', 'codigo_subcircuito', '_hora_num', '_dia_semana', '_mes', '_anio']


In [54]:
# =========================
# ENTRENAR RANDOM FOREST E IMPUTAR
# =========================

mask_original = df["codigo_iccs_original"].str.upper().isin({"SIN_DATO", "SIN DATOS", "TRUE"}) == False
mask_sintetico = df["codigo_iccs_original"].str.upper().isin({"SIN_DATO", "SIN DATOS", "TRUE"})

X_train = df_model[mask_original].values
X_pred  = df_model[mask_sintetico].values

# Codificar variable objetivo
le_target = LabelEncoder()
y_train = le_target.fit_transform(df.loc[mask_original, col_codigo].astype(str))

print(f"Registros de entrenamiento  : {X_train.shape[0]}")
print(f"Registros a imputar         : {X_pred.shape[0]}")
print(f"Clases únicas de codigo_iccs: {len(le_target.classes_)}")

# Entrenar el modelo
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("\nModelo entrenado correctamente.")

# Predecir para los SIN_DATO
y_pred_codigos = le_target.inverse_transform(rf.predict(X_pred))
df.loc[mask_sintetico, col_codigo] = y_pred_codigos

print("\nDistribución de códigos imputados (top 10):")
print(pd.Series(y_pred_codigos).value_counts().head(10))

Registros de entrenamiento  : 54677
Registros a imputar         : 22183
Clases únicas de codigo_iccs: 180

Modelo entrenado correctamente.

Distribución de códigos imputados (top 10):
060124.01    2059
020112.05     744
080690.02     693
040121.03     627
080900.02     614
020190.01     601
110900.04     517
050290.01     513
020510.00     495
020111.11     491
Name: count, dtype: int64


## 5. Creación de columnas `delitos` y `tipo_dato`

In [55]:
# =========================
# CREAR COLUMNA DELITOS
# Mapea codigo_iccs a su descripción; si no existe en el catálogo, queda 'NO ESPECIFICADO'
# =========================
df["delitos"] = df[col_codigo].map(mapa_delitos).fillna("NO ESPECIFICADO")

In [56]:
# =========================
# CREAR COLUMNA tipo_dato
# ORIGINAL  : el codigo_iccs vino del dataset sin modificación
# SINTETICO : era SIN_DATO y fue imputado por el modelo Random Forest
# =========================
df["tipo_dato"] = np.where(
    df["codigo_iccs_original"].str.upper().isin({"SIN_DATO", "SIN DATOS"}),
    "SINTETICO",
    "ORIGINAL"
)

print(df["tipo_dato"].value_counts(dropna=False))

tipo_dato
ORIGINAL     56045
SINTETICO    20815
Name: count, dtype: int64


In [57]:
# =========================
# ORDENAR COLUMNAS
# Resultado: codigo_iccs | delitos | tipo_dato | resto...
# =========================
cols = df.columns.tolist()

for c in ["delitos", "tipo_dato"]:
    if c in cols:
        cols.remove(c)

pos_codigo = cols.index(col_codigo) + 1
cols.insert(pos_codigo, "delitos")
cols.insert(pos_codigo + 1, "tipo_dato")

df = df[cols]

In [58]:
# =========================
# ELIMINAR COLUMNAS AUXILIARES
# =========================
cols_aux = ["codigo_iccs_original", "_fecha_dt", "_hora_num", "_dia_semana", "_mes", "_anio"]
df.drop(columns=[c for c in cols_aux if c in df.columns], inplace=True)

## 6. Imputación de coordenadas geográficas

Los registros con `NO_APLICA` en `latitud` / `longitud` (~67.000 filas) reciben las coordenadas
del centroide de su subcircuito, calculado a partir de los ~9.800 registros con coordenadas reales.
Para el subcircuito sin ningún registro real se usa el centroide del circuito al que pertenece.
La columna `flag_coord` registra la procedencia de cada coordenada para garantizar trazabilidad.

In [59]:
# =========================
# PREPARAR COLUMNAS DE COORDENADAS
# Las coordenadas vienen como texto con coma decimal (ej. -2,090978673)
# y el valor NO_APLICA marca los registros sin dato real.
# =========================
INVALIDOS_COORD = {"NO_APLICA", "", "SIN_DATO", "NAN", "NONE"}

def parse_coord(val):
    v = str(val).strip()
    if v.upper() in INVALIDOS_COORD:
        return float("nan")
    return pd.to_numeric(v.replace(",", "."), errors="coerce")

df["latitud"]  = df["latitud"].apply(parse_coord)
df["longitud"] = df["longitud"].apply(parse_coord)

mask_sin_coord = df["latitud"].isna()
print(f"Con coordenadas reales : {(~mask_sin_coord).sum()}")
print(f"Sin coordenadas        : {mask_sin_coord.sum()}")

Con coordenadas reales : 9814
Sin coordenadas        : 67046


In [60]:
# =========================
# CALCULAR CENTROIDES POR SUBCIRCUITO Y CIRCUITO
# =========================
df_con_coord = df[~mask_sin_coord].copy()

centroide_subcircuito = (
    df_con_coord
    .groupby("codigo_subcircuito")[["latitud", "longitud"]]
    .mean()
    .rename(columns={"latitud": "lat_sub", "longitud": "lon_sub"})
)

# Fallback para el subcircuito 09D01C06S03 que no tiene ninguna coord real
centroide_circuito = (
    df_con_coord
    .groupby("codigo_circuito")[["latitud", "longitud"]]
    .mean()
    .rename(columns={"latitud": "lat_cir", "longitud": "lon_cir"})
)

print(f"Subcircuitos con centroide : {len(centroide_subcircuito)}")
print(f"Circuitos con centroide    : {len(centroide_circuito)}")

Subcircuitos con centroide : 239
Circuitos con centroide    : 57


In [61]:
# =========================
# ASIGNAR COORDENADAS Y CREAR flag_coord
# =========================
df = df.join(centroide_subcircuito, on="codigo_subcircuito")
df = df.join(centroide_circuito,    on="codigo_circuito")

df.loc[mask_sin_coord, "latitud"] = (
    df.loc[mask_sin_coord, "lat_sub"]
    .fillna(df.loc[mask_sin_coord, "lat_cir"])
)
df.loc[mask_sin_coord, "longitud"] = (
    df.loc[mask_sin_coord, "lon_sub"]
    .fillna(df.loc[mask_sin_coord, "lon_cir"])
)

# flag_coord: REAL = coordenada original del MDI, IMPUTADO = centroide del subcircuito
df["flag_coord"] = np.where(mask_sin_coord, "IMPUTADO", "REAL")

df.drop(columns=["lat_sub", "lon_sub", "lat_cir", "lon_cir"], inplace=True)

print("flag_coord:")
print(df["flag_coord"].value_counts())
print(f"\nCoordenadas nulas restantes: {df['latitud'].isna().sum()}")

flag_coord:
flag_coord
IMPUTADO    67046
REAL         9814
Name: count, dtype: int64

Coordenadas nulas restantes: 0


## 7. Verificación final y exportación

In [62]:
# =========================
# VERIFICACIÓN FINAL
# =========================
print("=== Resumen final ===")
print(f"Filas totales          : {len(df)}")
print(f"Columnas               : {len(df.columns)}")
print("\ntipo_dato:")
print(df["tipo_dato"].value_counts())

sin_dato_restantes = df[col_codigo].str.upper().isin({"SIN_DATO", "SIN DATOS"}).sum()
print(f"\nSIN_DATO restantes en codigo_iccs: {sin_dato_restantes}")

print("\nflag_coord:")
print(df["flag_coord"].value_counts())
print(f"Coordenadas nulas restantes: {df['latitud'].isna().sum()}")

print("\nTop 5 delitos imputados (SINTETICO):")
print(df[df["tipo_dato"] == "SINTETICO"]["delitos"].value_counts().head(5))

print("\nMuestra de las primeras filas:")
df[[col_codigo, "delitos", "tipo_dato", "latitud", "longitud", "flag_coord"]].head(10)

=== Resumen final ===
Filas totales          : 76860
Columnas               : 39

tipo_dato:
tipo_dato
ORIGINAL     56045
SINTETICO    20815
Name: count, dtype: int64

SIN_DATO restantes en codigo_iccs: 0

flag_coord:
flag_coord
IMPUTADO    67046
REAL         9814
Name: count, dtype: int64
Coordenadas nulas restantes: 0

Top 5 delitos imputados (SINTETICO):
delitos
DELITOS CONTRA EL DERECHO A LA PROPIEDAD                                                         4510
DELITOS POR LA PRODUCCIÓN O TRÁFICO ILÍCITO DE SUSTANCIAS CATALOGADAS SUJETAS A FISCALIZACIÓN    2415
DELITOS CONTRA LA EFICIENCIA DE LA ADMINISTRACIÓN PÚBLICA                                        2016
CONTRAVENCIONES                                                                                  1275
DELITOS CONTRA LA TUTELA JUDICIAL EFECTIVA                                                        958
Name: count, dtype: int64

Muestra de las primeras filas:


,codigo_iccs,delitos,tipo_dato,latitud,longitud,flag_coord
0,020510.00,DELITOS CONTRA EL DERECHO A LA PROPIEDAD,SINTETICO,-2.090979,-79.921294,REAL
1,010102.01,DELITOS CONTRA LA INVIOLABILIDAD DE LA VIDA,ORIGINAL,-2.094478,-79.927189,REAL
2,010102.03,DELITOS CONTRA EL DERECHO A LA PROPIEDAD,SINTETICO,-2.996068,-79.792878,REAL
3,060124.01,DELITOS POR LA PRODUCCIÓN O TRÁFICO ILÍCITO DE...,ORIGINAL,-2.186058,-79.887123,REAL
4,060124.01,DELITOS POR LA PRODUCCIÓN O TRÁFICO ILÍCITO DE...,ORIGINAL,-2.252308,-79.905637,REAL
5,080900.02,DELITOS CONTRA LA EFICIENCIA DE LA ADMINISTRAC...,ORIGINAL,-2.139356,-79.908149,REAL
6,060124.01,DELITOS POR LA PRODUCCIÓN O TRÁFICO ILÍCITO DE...,ORIGINAL,-2.251310,-79.905270,REAL
7,060124.01,DELITOS POR LA PRODUCCIÓN O TRÁFICO ILÍCITO DE...,ORIGINAL,-2.237910,-79.910722,REAL
8,090111.03,DELITOS CONTRA LA SEGURIDAD PÚBLICA,ORIGINAL,-2.201751,-79.909852,REAL
9,090111.03,DELITOS CONTRA LA SEGURIDAD PÚBLICA,ORIGINAL,-2.201751,-79.909809,REAL


In [63]:
# =========================
# GUARDAR DATASET LIMPIO
# =========================
df.to_csv(archivo_salida, sep=";", index=False, encoding="utf-8-sig")
print("Archivo guardado en:")
print(archivo_salida)

Archivo guardado en:
C:\Users\jhono\Downloads\Proyecto Titulación\CSV\mdi_detenidosaprehendidos_guayaquil_2019_2025_con_delitos_y_tipo_dato.csv


In [64]:
# =========================
# GUARDAR CATÁLOGO (OPCIONAL)
# =========================
from pathlib import Path

if guardar_catalogo:
    ruta_catalogo = Path(archivo_catalogo)
    if not ruta_catalogo.parent.exists():
        ruta_catalogo = Path(archivo_salida).with_name("catalogo_iccs_generado.csv")
        print("Ruta de catálogo no disponible. Usando ruta local:")
        print(str(ruta_catalogo))
    ruta_catalogo.parent.mkdir(parents=True, exist_ok=True)
    catalogo.to_csv(ruta_catalogo, sep=";", index=False, encoding="utf-8-sig")
    print("Catálogo guardado en:")
    print(str(ruta_catalogo))

Catálogo guardado en:
C:\Users\jhono\Downloads\Proyecto Titulación\CSV\catalogo_iccs_generado.csv
